# COMP2322 Multi Threaded Web Server Test Notebook
This notebook separates each required test into an independent unit so each result can be captured clearly for the report.

### Cell 1
Purpose: Prepare the test image, define the target server address, and start `server.py` in the background.

In [2]:
import subprocess
import time
import requests
import socket
import threading
import os

PORT = 8848
BASE_URL = f"http://127.0.0.1:{PORT}"

# Update file modification time to now
os.utime('webs/index.html', (time.time(), time.time()))
print("Updated index.html modification time to current time")

# Kill existing process on port 8848 (Windows & Mac/Linux compatible)
print("Checking for existing server on port 8848...")
try:
    # Try Mac/Linux
    result = subprocess.run(["lsof", "-ti", f":{PORT}"], capture_output=True, text=True)
    if result.stdout.strip():
        pids = result.stdout.strip().split('\n')
        for pid in pids:
            subprocess.run(["kill", "-9", pid], capture_output=True)
            print(f"Killed existing process: {pid}")
except FileNotFoundError:
    # Windows: use netstat to find PID
    result = subprocess.run(f"netstat -ano | findstr :{PORT}", shell=True, capture_output=True, text=True)
    if result.stdout.strip():
        lines = result.stdout.strip().split('\n')
        for line in lines:
            parts = line.split()
            if len(parts) >= 5:
                pid = parts[-1]
                subprocess.run(f"taskkill /F /PID {pid}", shell=True, capture_output=True)
                print(f"Killed existing process: {pid}")
except Exception as e:
    print(f"No existing process or error: {e}")

time.sleep(1)

# Start new server
print("Starting new server process...")
server_process = subprocess.Popen(["python", "server.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print(f"Server is ready. Test target: {BASE_URL}")

# Store process for later termination
globals()['server_process'] = server_process

Updated index.html modification time to current time
Checking for existing server on port 8848...
Killed existing process: 13216
Starting new server process...
Server is ready. Test target: http://127.0.0.1:8848


### Cell 2
Purpose: Test a normal GET request for a text file and verify the `200 OK` response.

In [15]:
print("--- Test Item: GET text file and 200 OK ---")
url = f"{BASE_URL}/index.html"
response = requests.get(url)

print(f"Request URL: {url}")
print(f"Status code: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print(f"First 50 characters of response body:\n{response.text[:50]}...")

--- Test Item: GET text file and 200 OK ---
Request URL: http://127.0.0.1:8848/index.html
Status code: 200
Content-Type: text/html
First 50 characters of response body:
<html>
<body>
    <h1>BIBILABO</h1>
    <p>BABABOI...


### Cell 3
Purpose: Test a GET request for an image file and verify that binary content is returned correctly.

In [16]:
print("--- Test Item: GET image file ---")
url = f"{BASE_URL}/test.jpg"
response = requests.get(url)

print(f"Request URL: {url}")
print(f"Status code: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print(f"Returned image size: {len(response.content)} bytes")

--- Test Item: GET image file ---
Request URL: http://127.0.0.1:8848/test.jpg
Status code: 200
Content-Type: image/jpeg
Returned image size: 46161 bytes


### Cell 4
Purpose: Test the `HEAD` method and confirm that headers are returned without a response body.

In [17]:
print("--- Test Item: HEAD method ---")
url = f"{BASE_URL}/index.html"
response = requests.head(url)

print(f"Request URL: {url}")
print(f"Status code: {response.status_code}")
print("Returned headers:")
for k, v in response.headers.items():
    print(f"  {k}: {v}")
print(f"Body length: {len(response.text) if response.text else 0}")

--- Test Item: HEAD method ---
Request URL: http://127.0.0.1:8848/index.html
Status code: 200
Returned headers:
  Content-Type: text/html
  Content-Length: 71
  Last-Modified: Mon, 20 Apr 2026 14:07:39 GMT
  Connection: keep-alive
Body length: 0


### Cell 5
Purpose: Test a request for a missing file and verify the `404 File Not Found` response.

In [18]:
print("--- Test Item: 404 File Not Found ---")
url = f"{BASE_URL}/this_file_does_not_exist.html"
response = requests.get(url)

print(f"Request URL: {url}")
print(f"Status code: {response.status_code}")
print(f"Returned content:\n{response.text[:100].strip()}...")

--- Test Item: 404 File Not Found ---
Request URL: http://127.0.0.1:8848/this_file_does_not_exist.html
Status code: 404
Returned content:
<html>
    <head><title>404 Not Found</title></head>
    <body>
        <h1>404 Not Found</h1>...


### Cell 6
Purpose: Test path traversal protection and verify the `403 Forbidden` response.

In [19]:
print("--- Test Item: 403 Forbidden ---")
url = f"{BASE_URL}/../server.py"
response = requests.get(url)

print(f"Request URL: {url}")
print(f"Status code: {response.status_code}")
print(f"Returned content:\n{response.text[:100].strip()}...")

--- Test Item: 403 Forbidden ---
Request URL: http://127.0.0.1:8848/../server.py
Status code: 403
Returned content:
<html>
    <head><title>403 Forbidden</title></head>
    <body>
        <h1>403 Forbidden</h1>...


### Cell 7
Purpose: Send an invalid raw socket request and verify the `400 Bad Request` response.

In [3]:
print("--- Test Item: 400 Bad Request ---")
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.connect(("127.0.0.1", PORT))
    bad_request = b"HELLO THIS IS NOT HTTP\r\n\r\n"
    s.send(bad_request)
    raw_response = s.recv(1024).decode("utf-8", errors="ignore")
    s.close()

    status_line = raw_response.split("\r\n")[0]
    print("Sent invalid request: HELLO THIS IS NOT HTTP")
    print(f"Server response status line: {status_line}")
except Exception as e:
    print(f"Connection failed: {e}")

--- Test Item: 400 Bad Request ---
Sent invalid request: HELLO THIS IS NOT HTTP
Server response status line: HTTP/1.1 400 Bad Request


### Cell 8
Purpose: Test `Last-Modified` and `304 Not Modified` by sending a second request with `If-Modified-Since`.

In [21]:
print("--- Test Item: 304 Not Modified and Last-Modified ---")
url = f"{BASE_URL}/index.html"

r1 = requests.get(url)
last_modified_date = r1.headers.get("Last-Modified")
print(f"First request status code: {r1.status_code}")
print(f"Returned Last-Modified: {last_modified_date}")

headers = {"If-Modified-Since": last_modified_date}
r2 = requests.get(url, headers=headers)
print(f"\nSecond request header: If-Modified-Since = {last_modified_date}")
print(f"Second request status code: {r2.status_code}")

--- Test Item: 304 Not Modified and Last-Modified ---
First request status code: 200
Returned Last-Modified: Mon, 20 Apr 2026 14:07:39 GMT

Second request header: If-Modified-Since = Mon, 20 Apr 2026 14:07:39 GMT
Second request status code: 304


### Cell 9
Purpose: Test how the server handles the `Connection` header for both `keep-alive` and `close`.

In [22]:
print("--- Test Item: Connection header ---")
url = f"{BASE_URL}/index.html"

headers_keep = {"Connection": "keep-alive"}
r_keep = requests.get(url, headers=headers_keep)
print(f"Sent header: {headers_keep}")
print(f"Returned Connection header: {r_keep.headers.get('Connection')}")
print()

headers_close = {"Connection": "close"}
r_close = requests.get(url, headers=headers_close)
print(f"Sent header: {headers_close}")
print(f"Returned Connection header: {r_close.headers.get('Connection')}")

--- Test Item: Connection header ---
Sent header: {'Connection': 'keep-alive'}
Returned Connection header: keep-alive

Sent header: {'Connection': 'close'}
Returned Connection header: close


### Cell 10
Purpose: Launch multiple requests at the same time to demonstrate multi threaded concurrency.

In [23]:
print("--- Test Item: Multi threaded concurrency ---")

def fetch_url(thread_id):
    try:
        r = requests.get(f"{BASE_URL}/index.html")
        print(f"Thread {thread_id} succeeded | Status code: {r.status_code}")
    except Exception as e:
        print(f"Thread {thread_id} failed | Error: {e}")

threads = []
for i in range(1, 6):
    t = threading.Thread(target=fetch_url, args=(i,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("All concurrent requests have completed.")

--- Test Item: Multi threaded concurrency ---
Thread 1 succeeded | Status code: 200
Thread 2 succeeded | Status code: 200
Thread 3 succeeded | Status code: 200
Thread 5 succeeded | Status code: 200
Thread 4 succeeded | Status code: 200
All concurrent requests have completed.


### Cell 11
Purpose: Display recent log records to show that the server recorded the test activity.

In [24]:
print("--- Test Item: Log file records ---")
log_path = "logs/server.log"

with open(log_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"Total number of log lines: {len(lines)}")
print()
print("Most recent 10 log records:")
for line in lines[-10:]:
    print(line.strip())

--- Test Item: Log file records ---
Total number of log lines: 201

Most recent 10 log records:
127.0.0.1 | 2026-04-20 22:07:42 | GET /../server.py | 403
127.0.0.1 | 2026-04-20 22:07:42 | UNKNOWN UNKNOWN | 400
127.0.0.1 | 2026-04-20 22:07:42 | GET /index.html | 200
127.0.0.1 | 2026-04-20 22:07:42 | GET /index.html | 304
127.0.0.1 | 2026-04-20 22:07:42 | GET /index.html | 200
127.0.0.1 | 2026-04-20 22:07:42 | GET /index.html | 200
127.0.0.1 | 2026-04-20 22:07:42 | GET /index.html | 200
127.0.0.1 | 2026-04-20 22:07:42 | GET /index.html | 200
127.0.0.1 | 2026-04-20 22:07:42 | GET /index.html | 200
127.0.0.1 | 2026-04-20 22:07:42 | GET /index.html | 200


### Cell X
Purpose: Stop the background server process and release the port after all tests are complete.

In [ ]:
print("--- Test Item: Status Dashboard ---")
url = f"{BASE_URL}/status"

response = requests.get(url)

print(f"Request URL: {url}")
print(f"Status Code: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print(f"Content-Length: {len(response.content)} bytes")
print("\n--- Dashboard Preview (first 800 characters) ---")
print(response.text[:800])

with open('status_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(response.text)
print("\n[Info] Dashboard saved to status_dashboard.html")

--- Test Item: Status Dashboard ---
Request URL: http://127.0.0.1:8848/status
Status Code: 200
Content-Type: text/html
Content-Length: 5354 bytes

--- Dashboard Preview (first 800 characters) ---

<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Server Status</title>
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
            background: #f5f5f5;
        }
        h1 { color: #333; border-bottom: 3px solid #4CAF50; padding-bottom: 10px; }
        h2 { color: #555; margin-top: 30px; }
        .card {
            background: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        table {
            width: 100%;
            border-collapse: collapse;
            margin-top: 10px;
        }
        th, td {
            

[I

### Cell 12
Purpose: Stop the background server process and release the port after all tests are complete.

In [4]:
print("--- Terminating server process ---")
server_process.terminate()
print(f"Port {PORT} has been released. Testing is complete.")

--- Terminating server process ---
Port 8848 has been released. Testing is complete.
